# Nexus — HyperShelf: Localization Mismatch Profiling
## Objective 1b — Hyper-Local Demand Analysis Across All Stores
> **What this notebook does:** Identifies products and categories whose sales performance
> at a specific store is significantly below (or above) the regional average.
> These mismatches signal wrong assortments, phantom inventory, or localized demand gaps.
> Output: ranked mismatch table + store-level localization score + actionable flags.

---
## Section 0 — Imports & Config

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)

DATA_DIR   = Path('/Users/shashi/retail-ai-project/data/raw/output/csv')
OUTPUT_DIR = Path('/Users/shashi/retail-ai-project/data/processed/nexus/localization')
OUTPUT_DIR.mkdir(exist_ok=True)

NOW   = datetime.now()

# ── MISMATCH THRESHOLDS ───────────────────────────────────────────
# A store-category is 'mismatched' if its avg revenue/SKU is this far below regional avg
MISMATCH_PCT_THRESHOLD = 0.40   # 40% below regional average = flagged
EXTREME_PCT_THRESHOLD  = 0.70   # 70% below = extreme mismatch
MIN_SKU_COUNT          = 3      # ignore categories with fewer than 3 SKUs

print('='*62)
print('  NEXUS | LOCALIZATION MISMATCH PROFILING')
print('='*62)
print(f'  Generated : {NOW.strftime("%A, %B %d, %Y at %I:%M %p")}')
print(f'  Mismatch threshold  : {MISMATCH_PCT_THRESHOLD*100:.0f}% below regional avg')
print(f'  Extreme threshold   : {EXTREME_PCT_THRESHOLD*100:.0f}% below regional avg')
print(f'  Min SKUs per cat    : {MIN_SKU_COUNT}')
print('='*62)


  NEXUS | LOCALIZATION MISMATCH PROFILING
  Generated : Saturday, April 25, 2026 at 01:25 PM
  Mismatch threshold  : 40% below regional avg
  Extreme threshold   : 70% below regional avg
  Min SKUs per cat    : 3


---
## Section 1 — Load Data

In [2]:
stores   = pd.read_csv(DATA_DIR / 'stores.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')

print('Loading sales...')
chunks = []
for chunk in pd.read_csv(DATA_DIR / 'sales_transactions.csv',
                          parse_dates=['sale_date'], chunksize=500_000):
    grp = (chunk.groupby(['store_id','sku_id'])
               .agg(units_sold=('units_sold','sum'), revenue=('revenue','sum'),
                    sale_days=('sale_date','nunique'))
               .reset_index())
    chunks.append(grp)

sales_agg = (pd.concat(chunks)
               .groupby(['store_id','sku_id'])
               .agg(units_sold=('units_sold','sum'), revenue=('revenue','sum'),
                    sale_days=('sale_days','sum'))
               .reset_index())

# Enrich with product category
sales_agg = sales_agg.merge(
    products[['sku_id','product_name','category','subcategory','unit_price']],
    on='sku_id', how='left')

# Enrich with store region
sales_agg = sales_agg.merge(
    stores[['store_id','store_name','city','state','region',
            'store_format','foot_traffic_tier']],
    on='store_id', how='left')

# Average daily sales per store-SKU
sales_agg['avg_daily_units'] = sales_agg['units_sold'] / sales_agg['sale_days'].replace(0, 1)
sales_agg['avg_daily_rev']   = sales_agg['revenue']    / sales_agg['sale_days'].replace(0, 1)

print(f'Store-SKU rows: {len(sales_agg):,}')
print(f'Unique stores : {sales_agg["store_id"].nunique():,}')
print(f'Unique SKUs   : {sales_agg["sku_id"].nunique():,}')
print(f'Categories    : {sales_agg["category"].nunique():,}')


Loading sales...
Store-SKU rows: 38,240
Unique stores : 478
Unique SKUs   : 1,265
Categories    : 13


---
## Section 2 — Regional Benchmark
> For each region + category combination, compute the average revenue per SKU per day.
> This is the benchmark every store-category pair will be compared against.

In [3]:
# ── Regional benchmark: avg revenue per SKU per day ──────────────
region_bench = (
    sales_agg.groupby(['region','category'])
    .agg(
        region_avg_daily_rev   = ('avg_daily_rev',   'mean'),
        region_median_daily_rev= ('avg_daily_rev',   'median'),
        region_std_daily_rev   = ('avg_daily_rev',   'std'),
        region_store_count     = ('store_id',         'nunique'),
        region_sku_count       = ('sku_id',           'nunique'),
    )
    .reset_index()
)

# ── Store-category level performance ─────────────────────────────
store_cat = (
    sales_agg.groupby(['store_id','region','category'])
    .agg(
        store_avg_daily_rev   = ('avg_daily_rev',  'mean'),
        store_total_revenue   = ('revenue',         'sum'),
        store_sku_count       = ('sku_id',          'nunique'),
        store_units_sold      = ('units_sold',       'sum'),
    )
    .reset_index()
)

# Merge benchmark in
store_cat = store_cat.merge(region_bench, on=['region','category'], how='left')

# ── Mismatch score: how far below regional avg ────────────────────
store_cat['rev_gap']        = store_cat['store_avg_daily_rev'] - store_cat['region_avg_daily_rev']
store_cat['rev_gap_pct']    = store_cat['rev_gap'] / store_cat['region_avg_daily_rev'].replace(0, np.nan)
store_cat['rev_gap_pct']    = store_cat['rev_gap_pct'].fillna(0)

# Z-score of this store vs regional distribution
store_cat['z_score'] = (
    (store_cat['store_avg_daily_rev'] - store_cat['region_avg_daily_rev']) /
    store_cat['region_std_daily_rev'].replace(0, np.nan)
).fillna(0)

# ── Apply mismatch flags ──────────────────────────────────────────
# Filter: only categories with enough SKUs
store_cat = store_cat[store_cat['store_sku_count'] >= MIN_SKU_COUNT].copy()

def mismatch_label(row):
    g = row['rev_gap_pct']
    if g <= -EXTREME_PCT_THRESHOLD:
        return 'EXTREME_MISMATCH'
    elif g <= -MISMATCH_PCT_THRESHOLD:
        return 'MISMATCH'
    elif g >= MISMATCH_PCT_THRESHOLD:
        return 'OVERPERFORMING'
    return 'ALIGNED'

store_cat['mismatch_flag'] = store_cat.apply(mismatch_label, axis=1)

flag_counts = store_cat['mismatch_flag'].value_counts()
print('MISMATCH FLAGS ACROSS ALL STORES:')
for flag, cnt in flag_counts.items():
    pct = cnt / len(store_cat) * 100
    print(f'  {flag:<22}: {cnt:>6,} store-category combos ({pct:.1f}%)')
print(f'\n  Total store-category combos analyzed: {len(store_cat):,}')


MISMATCH FLAGS ACROSS ALL STORES:
  ALIGNED               :  2,798 store-category combos (49.5%)
  MISMATCH              :  1,378 store-category combos (24.4%)
  OVERPERFORMING        :  1,262 store-category combos (22.3%)
  EXTREME_MISMATCH      :    210 store-category combos (3.7%)

  Total store-category combos analyzed: 5,648


---
## Section 3 — Store Localization Score
> Each store gets a single localization score: % of its categories that are mismatched.
> High score = store assortment is well-aligned to local demand.
> Low score = store has many categories underperforming vs regional peers.

In [4]:
store_loc_score = (
    store_cat.groupby('store_id')
    .agg(
        total_cats         = ('category',       'count'),
        mismatch_cats      = ('mismatch_flag',  lambda x: (x.isin(['MISMATCH','EXTREME_MISMATCH'])).sum()),
        extreme_mismatch   = ('mismatch_flag',  lambda x: (x == 'EXTREME_MISMATCH').sum()),
        overperforming_cats= ('mismatch_flag',  lambda x: (x == 'OVERPERFORMING').sum()),
        avg_rev_gap_pct    = ('rev_gap_pct',    'mean'),
        total_revenue      = ('store_total_revenue','sum'),
    )
    .reset_index()
)

store_loc_score['mismatch_rate']       = store_loc_score['mismatch_cats'] / store_loc_score['total_cats']
store_loc_score['localization_score']  = (1 - store_loc_score['mismatch_rate']) * 100

# Merge store info
store_loc_score = store_loc_score.merge(
    stores[['store_id','store_name','city','state','region',
            'store_format','foot_traffic_tier']],
    on='store_id', how='left')

store_loc_score = store_loc_score.sort_values('localization_score').reset_index(drop=True)

print('STORE LOCALIZATION SCORES (worst first):')
print(f'{"Store":<10} {"Name":<28} {"City":<16} {"Region":<12}'
      f' {"Score":>8} {"Mismatched":>12} {"Extreme":>8} {"AvgGap%":>9}')
print('-'*110)
for _, r in store_loc_score.head(20).iterrows():
    print(
        f'{r["store_id"]:<10}'
        f' {str(r["store_name"])[:26]:<28}'
        f' {str(r["city"])[:14]:<16}'
        f' {str(r["region"])[:10]:<12}'
        f' {r["localization_score"]:>7.1f}%'
        f' {int(r["mismatch_cats"]):>7}/{int(r["total_cats"]):<4}'
        f' {int(r["extreme_mismatch"]):>8}'
        f' {r["avg_rev_gap_pct"]*100:>8.1f}%'
    )


STORE LOCALIZATION SCORES (worst first):
Store      Name                         City             Region          Score   Mismatched  Extreme   AvgGap%
--------------------------------------------------------------------------------------------------------------
S0427      MapleCenter San Antonio      San Antonio      Southwest        0.0%      10/10          1    -58.1%
S0235      MetroDepot Pittsburgh        Pittsburgh       Southeast        0.0%      12/12          1    -52.7%
S0068      GreenFresh San Diego         San Diego        Southwest        0.0%      12/12          2    -58.0%
S0301      GoldenCorner New York        New York         Southwest        0.0%      12/12          2    -59.9%
S0361      BrightStop New York          New York         Northwest        0.0%      11/11          2    -55.4%
S0167      MaplePoint Honolulu          Honolulu         West             7.7%      12/13          3    -57.2%
S0153      RiverMarket Sacramento       Sacramento       Northeast     

---
## Section 4 — Top Mismatch Categories Across the Network
> Which categories are consistently mismatched across the most stores?

In [5]:
cat_mismatch_network = (
    store_cat[store_cat['mismatch_flag'].isin(['MISMATCH','EXTREME_MISMATCH'])]
    .groupby('category')
    .agg(
        stores_with_mismatch = ('store_id', 'nunique'),
        avg_rev_gap_pct      = ('rev_gap_pct', 'mean'),
        total_rev_gap        = ('rev_gap', 'sum'),
        extreme_count        = ('mismatch_flag', lambda x: (x=='EXTREME_MISMATCH').sum()),
    )
    .reset_index()
    .sort_values('stores_with_mismatch', ascending=False)
)

print('TOP MISMATCHED CATEGORIES (most stores affected):')
print(f'{"Category":<22} {"Stores Affected":>16} {"Avg Gap%":>10}'
      f' {"Extreme Count":>14}')
print('-'*70)
for _, r in cat_mismatch_network.head(15).iterrows():
    print(
        f'{str(r["category"]):<22}'
        f' {int(r["stores_with_mismatch"]):>16,}'
        f' {r["avg_rev_gap_pct"]*100:>9.1f}%'
        f' {int(r["extreme_count"]):>14,}'
    )


TOP MISMATCHED CATEGORIES (most stores affected):
Category                Stores Affected   Avg Gap%  Extreme Count
----------------------------------------------------------------------
Personal Care                       136     -55.2%             12
Beverages                           131     -53.6%              5
Snacks                              129     -54.4%              9
Dairy & Eggs                        128     -55.6%             14
Canned & Pantry                     127     -56.7%             18
Meat & Seafood                      126     -58.5%             16
Produce                             126     -58.7%             25
Bakery                              120     -56.1%             18
Frozen Foods                        119     -55.6%             11
Baby & Infant                       118     -60.5%             28
Pet Supplies                        115     -57.7%             20
Household                           112     -56.3%             16
Health & Wellness    

---
## Section 5 — Store-Level Drill Down
> For any given store, shows every mismatched category with its top underperforming SKUs.

In [6]:
def store_localization_report(store_id, store_cat, sales_agg, top_n_skus=5):
    """Print full localization mismatch report for one store."""
    s_data = store_cat[store_cat['store_id'] == store_id].copy()
    s_info = stores[stores['store_id'] == store_id]
    if len(s_data) == 0:
        print(f'No data for {store_id}')
        return

    sname  = s_info.iloc[0]['store_name'] if len(s_info) > 0 else store_id
    scity  = s_info.iloc[0]['city'] if len(s_info) > 0 else ''
    sregion= s_info.iloc[0]['region'] if len(s_info) > 0 else ''

    mismatched = s_data[s_data['mismatch_flag'].isin(['MISMATCH','EXTREME_MISMATCH'])]
    score_row  = store_loc_score[store_loc_score['store_id'] == store_id]
    score      = score_row.iloc[0]['localization_score'] if len(score_row) > 0 else 0

    print('='*72)
    print(f'  LOCALIZATION REPORT — {store_id} {sname}')
    print(f'  {scity} | {sregion}')
    print(f'  Localization Score: {score:.1f} / 100')
    print(f'  {len(mismatched)} categories mismatched out of {len(s_data)}')
    print('='*72)

    for _, cat_row in mismatched.sort_values('rev_gap_pct').iterrows():
        cat    = cat_row['category']
        flag   = cat_row['mismatch_flag']
        gap    = cat_row['rev_gap_pct'] * 100
        store_rev = cat_row['store_avg_daily_rev']
        reg_rev   = cat_row['region_avg_daily_rev']

        print(f'\n  [{flag}] {cat}')
        print(f'    Store avg: ${store_rev:.2f}/day per SKU  |  '
              f'Regional avg: ${reg_rev:.2f}/day per SKU  |  '
              f'Gap: {gap:.1f}%')

        # Top underperforming SKUs in this category at this store
        sku_sub = sales_agg[
            (sales_agg['store_id'] == store_id) &
            (sales_agg['category'] == cat)
        ].copy()

        # Regional avg for these SKUs
        sku_reg = (
            sales_agg[sales_agg['category'] == cat]
            .groupby('sku_id')['avg_daily_rev'].mean()
            .reset_index()
            .rename(columns={'avg_daily_rev':'reg_sku_avg'})
        )
        sku_sub = sku_sub.merge(sku_reg, on='sku_id', how='left')
        sku_sub['sku_gap_pct'] = (
            (sku_sub['avg_daily_rev'] - sku_sub['reg_sku_avg']) /
            sku_sub['reg_sku_avg'].replace(0, np.nan)
        ).fillna(0)
        worst_skus = sku_sub.nsmallest(top_n_skus, 'sku_gap_pct')

        print(f'    Worst performing SKUs:')
        for _, s in worst_skus.iterrows():
            pname = str(s.get('product_name',''))[:32]
            print(f'      {s["sku_id"]}  {pname:<34}'
                  f'  Store: ${s["avg_daily_rev"]:.2f}/d'
                  f'  Region: ${s["reg_sku_avg"]:.2f}/d'
                  f'  Gap: {s["sku_gap_pct"]*100:.1f}%')


# Run for top 3 most mismatched stores
for sid in store_loc_score.head(3)['store_id']:
    store_localization_report(sid, store_cat, sales_agg)
    print()


  LOCALIZATION REPORT — S0427 MapleCenter San Antonio
  San Antonio | Southwest
  Localization Score: 0.0 / 100
  10 categories mismatched out of 10

  [EXTREME_MISMATCH] Meat & Seafood
    Store avg: $87.20/day per SKU  |  Regional avg: $301.90/day per SKU  |  Gap: -71.1%
    Worst performing SKUs:
      P00871  CoastalFarms Light Pork 6-Pack      Store: $70.79/d  Region: $511.71/d  Gap: -86.2%
      P00901  EverFresh Light Chicken 6-Pack      Store: $85.26/d  Region: $220.90/d  Gap: -61.4%
      P00863  TrueGrain Extra Fresh Pork 6-Pac    Store: $90.40/d  Region: $221.48/d  Gap: -59.2%
      P00894  PureStart Lite Seafood 1L           Store: $177.32/d  Region: $389.58/d  Gap: -54.5%
      P00826  BrightBasket Premium Seafood 6-P    Store: $24.01/d  Region: $50.75/d  Gap: -52.7%

  [MISMATCH] Frozen Foods
    Store avg: $99.90/day per SKU  |  Regional avg: $276.07/day per SKU  |  Gap: -63.8%
    Worst performing SKUs:
      P00629  SunValley Ultra Pizza 150g          Store: $83.56/d  

---
## Section 6 — Localization Visualizations

In [7]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f'Localization Mismatch Analysis — {NOW.strftime("%B %d, %Y")} — {len(store_loc_score)} Stores',
             fontsize=12, fontweight='bold')

# Panel 1: Localization score distribution
axes[0,0].hist(store_loc_score['localization_score'], bins=20, color='#0D9488', edgecolor='white')
axes[0,0].axvline(store_loc_score['localization_score'].mean(), color='#DC2626',
                  linestyle='--', linewidth=2, label=f'Mean: {store_loc_score["localization_score"].mean():.1f}')
axes[0,0].set_title('Distribution of Localization Scores')
axes[0,0].set_xlabel('Localization Score (100 = fully aligned)')
axes[0,0].set_ylabel('Stores')
axes[0,0].legend()

# Panel 2: Top mismatched categories (bar)
top_cats = cat_mismatch_network.head(12)
axes[0,1].barh(top_cats['category'], top_cats['stores_with_mismatch'],
               color='#DC2626', alpha=0.8)
axes[0,1].set_title('Most Mismatched Categories\n(# stores below threshold)')
axes[0,1].set_xlabel('Stores with Mismatch')

# Panel 3: Localization score by region
# store_loc_score already has 'region' from the merge in Section 3
reg_avg   = store_loc_score.groupby('region')['localization_score'].mean().sort_values(ascending=True)
colors3   = ['#DC2626' if v < 70 else '#D97706' if v < 85 else '#059669' for v in reg_avg.values]
axes[1,0].barh(reg_avg.index, reg_avg.values, color=colors3)
axes[1,0].axvline(100, color='gray', linestyle='--', linewidth=1)
axes[1,0].set_title('Avg Localization Score by Region')
axes[1,0].set_xlabel('Localization Score')
axes[1,0].set_xlim(0, 110)
for i, v in enumerate(reg_avg.values):
    axes[1,0].text(v+0.5, i, f'{v:.1f}', va='center', fontsize=8)

# Panel 4: Mismatch flag counts
flag_order  = ['EXTREME_MISMATCH','MISMATCH','ALIGNED','OVERPERFORMING']
flag_colors = ['#DC2626','#D97706','#059669','#0D9488']
flag_vals   = [flag_counts.get(f, 0) for f in flag_order]
axes[1,1].pie(flag_vals, labels=flag_order, colors=flag_colors,
              autopct='%1.1f%%', startangle=90,
              wedgeprops={'edgecolor':'white','linewidth':2})
axes[1,1].set_title('Store-Category Mismatch Flags\nNetwork-wide')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'localization_overview.png', bbox_inches='tight', dpi=130)
plt.show()
print('Saved: localization_overview.png')


Saved: localization_overview.png


---
## Section 7 — Export All Localization Outputs

In [8]:
# Full store-category mismatch table
store_cat.insert(0, 'report_time', NOW.strftime('%Y-%m-%d %H:%M:%S'))
store_cat.to_csv(OUTPUT_DIR / 'store_category_mismatch.csv', index=False)
print(f'store_category_mismatch.csv    — {len(store_cat):,} rows')

# Store localization scores
store_loc_score.insert(0, 'report_time', NOW.strftime('%Y-%m-%d %H:%M:%S'))
store_loc_score.to_csv(OUTPUT_DIR / 'store_localization_scores.csv', index=False)
print(f'store_localization_scores.csv  — {len(store_loc_score)} stores')

# Mismatched only
mismatched_only = store_cat[
    store_cat['mismatch_flag'].isin(['MISMATCH','EXTREME_MISMATCH'])
].sort_values(['store_id','rev_gap_pct'])
mismatched_only.to_csv(OUTPUT_DIR / 'mismatched_store_categories.csv', index=False)
print(f'mismatched_store_categories.csv— {len(mismatched_only):,} rows')

# Category network mismatch summary
cat_mismatch_network.to_csv(OUTPUT_DIR / 'category_network_mismatch.csv', index=False)
print(f'category_network_mismatch.csv  — {len(cat_mismatch_network)} categories')

print(f'\nAll outputs saved to: {OUTPUT_DIR.resolve()}/')
print()
print('='*62)
print('  LOCALIZATION SUMMARY')
print('='*62)
print(f'  Total store-cat combos analyzed : {len(store_cat):,}')
print(f'  Extreme mismatches              : {flag_counts.get("EXTREME_MISMATCH",0):,}')
print(f'  Mismatches                      : {flag_counts.get("MISMATCH",0):,}')
print(f'  Aligned                         : {flag_counts.get("ALIGNED",0):,}')
print(f'  Overperforming                  : {flag_counts.get("OVERPERFORMING",0):,}')
print(f'  Worst store localization score  : {store_loc_score["localization_score"].min():.1f}')
print(f'  Best  store localization score  : {store_loc_score["localization_score"].max():.1f}')
print(f'  Avg   store localization score  : {store_loc_score["localization_score"].mean():.1f}')
print('='*62)


store_category_mismatch.csv    — 5,648 rows
store_localization_scores.csv  — 478 stores
mismatched_store_categories.csv— 1,588 rows
category_network_mismatch.csv  — 13 categories

All outputs saved to: /Users/shashi/retail-ai-project/data/processed/nexus/localization/

  LOCALIZATION SUMMARY
  Total store-cat combos analyzed : 5,648
  Extreme mismatches              : 210
  Mismatches                      : 1,378
  Aligned                         : 2,798
  Overperforming                  : 1,262
  Worst store localization score  : 0.0
  Best  store localization score  : 100.0
  Avg   store localization score  : 71.8
